## 1. Imports & Setup

In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LassoCV, LogisticRegressionCV
from sklearn.linear_model import (
    LinearRegression, LogisticRegression,
)

from sklearn.metrics import (
    r2_score, mean_absolute_error, root_mean_squared_error,
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [18]:
RESULTS_DIR = "results/lasso_results" # Create this directory

## 2. Load Data

In [19]:
df = pd.read_csv("data/cibil_score/cibil_score.csv")
df = df.drop(columns=["Unnamed: 0"])

# normalize column names
df.columns = [col.lower().strip() for col in df.columns]

print(df.shape)
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df.head()

(51336, 87)
Duplicate rows: 0


,prospectid,total_tl,tot_closed_tl,tot_active_tl,total_tl_opened_l6m,tot_tl_closed_l6m,pct_tl_open_l6m,pct_tl_closed_l6m,pct_active_tl,pct_closed_tl,...,pct_cc_enq_l6m_of_l12m,pct_pl_enq_l6m_of_ever,pct_cc_enq_l6m_of_ever,max_unsec_exposure_inpct,hl_flag,gl_flag,last_prod_enq2,first_prod_enq2,credit_score,approved_flag
0,1,5,4,1,0,0,0.000,0.0,0.200,0.800,...,0.0,0.0,0.0,13.333,1,0,PL,PL,696,P2
1,2,1,0,1,0,0,0.000,0.0,1.000,0.000,...,0.0,0.0,0.0,0.860,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,8,0,8,1,0,0.125,0.0,1.000,0.000,...,0.0,0.0,0.0,5741.667,1,0,ConsumerLoan,others,693,P2
3,4,1,0,1,1,0,1.000,0.0,1.000,0.000,...,0.0,0.0,0.0,9.900,0,0,others,others,673,P2
4,5,3,2,1,0,0,0.000,0.0,0.333,0.667,...,0.0,0.0,0.0,-99999.000,0,0,AL,AL,753,P1


In [20]:
df["approved_flag"].value_counts()

approved_flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64

## 3. Feature / Target Setup
* `X` -> all columns except `credit_score` and `approved_flag`
* `y_linear` -> `credit_score` (Linear Regression target)
* `y_multiclass` -> `approved_flag` (Logistic Regression, multiclass: P1/P2/P3/P4)
* `y_binary` -> `approved_flag` mapped to **1** for P1/P2 and **0** for P3/P4 (Logistic Regression, binary)

In [21]:
X = df.drop(columns=["approved_flag", "credit_score"])
y_linear = df["credit_score"].astype(float)
y_multiclass = df["approved_flag"].astype(str)

binary_map = {"P1": 1, "P2": 1, "P3": 0, "P4": 0}
y_binary = df["approved_flag"].map(binary_map)

assert y_binary.isna().sum() == 0, "approved_flag has values outside P1-P4"

print(y_multiclass.value_counts())
print(y_binary.value_counts())

approved_flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64
approved_flag
1    38002
0    13334
Name: count, dtype: int64


## 4. Train / Test Split
A single split on `X` is created and then reused (via the shared index) for every target so that every model family sees the exact same rows in train/test.

In [22]:
X_train, X_test, idx_train, idx_test = train_test_split(
    X, X.index, test_size=0.2, random_state=RANDOM_STATE
)

y_linear_train, y_linear_test = y_linear.loc[idx_train], y_linear.loc[idx_test]
y_multi_train, y_multi_test = y_multiclass.loc[idx_train], y_multiclass.loc[idx_test]
y_bin_train, y_bin_test = y_binary.loc[idx_train], y_binary.loc[idx_test]

print(X_train.shape, X_test.shape)

(41068, 85) (10268, 85)


## 5. Missing Value Handling
Domain-specific cleanup carried over/expanded from the original notebook. The dataset encodes several kinds of \"missing\" as the sentinel value `-99999`; the correct treatment depends on what the field means.

In [23]:
# 5a. Drop columns with very high missingness
high_missing = [c for c in ["cc_utilization", "pl_utilization"] if c in X_train.columns]
X_train = X_train.drop(columns=high_missing)
X_test = X_test.drop(columns=high_missing)

# 5b. Median-impute a handful of numeric fields where -99999 means "unknown"
median_columns = [
    "age_oldest_tl", "age_newest_tl", "pct_currentbal_all_tl", "time_since_recent_payment",
]
for col in median_columns:
    if col not in X_train.columns:
        continue
    X_train[col] = X_train[col].replace(-99999, np.nan)
    X_test[col] = X_test[col].replace(-99999, np.nan)
    train_median = X_train[col].median()
    X_train[col] = X_train[col].fillna(train_median)
    X_test[col] = X_test[col].fillna(train_median)  # use TRAIN median to avoid leakage

# 5c. Delinquency fields: -99999 means "never delinquent" -> 0
delinquency_columns = [
    "max_delinquency_level", "max_deliq_6mts", "max_deliq_12mts",
    "time_since_recent_deliquency", "time_since_first_deliquency",
]
delinquency_columns = [c for c in delinquency_columns if c in X_train.columns]
X_train[delinquency_columns] = X_train[delinquency_columns].replace(-99999, 0)
X_test[delinquency_columns] = X_test[delinquency_columns].replace(-99999, 0)

# 5d. Enquiry fields: -99999 means "no enquiry" -> 0
enquiry_columns = [
    "tot_enq", "cc_enq", "pl_enq", "cc_enq_l6m", "cc_enq_l12m",
    "pl_enq_l6m", "pl_enq_l12m", "enq_l3m", "enq_l6m", "enq_l12m",
]
enquiry_columns = [c for c in enquiry_columns if c in X_train.columns]
X_train[enquiry_columns] = X_train[enquiry_columns].replace(-99999, 0)
X_test[enquiry_columns] = X_test[enquiry_columns].replace(-99999, 0)

# 5e. time_since_recent_enq: -99999 = "no enquiry ever" -> longer than any observed gap,
# keep the "no enquiry" signal as its own binary flag (fit on TRAIN only, applied to both)
col = "time_since_recent_enq"
if col in X_train.columns:
    train_mask = X_train[col] == -99999
    test_mask = X_test[col] == -99999
    fill_value = X_train.loc[~train_mask, col].max() + 1

    X_train["no_enquiry_flag"] = train_mask.astype(int)
    X_test["no_enquiry_flag"] = test_mask.astype(int)

    X_train[col] = X_train[col].replace(-99999, fill_value)
    X_test[col] = X_test[col].replace(-99999, fill_value)

# 5f. max_unsec_exposure_inpct: -99999 = "no unsecured loan" -> 0%
col = "max_unsec_exposure_inpct"
if col in X_train.columns:
    X_train[col] = X_train[col].replace(-99999, 0)
    X_test[col] = X_test[col].replace(-99999, 0)

# 5g. Log-transform skewed income
if "netmonthlyincome" in X_train.columns:
    X_train["netmonthlyincome_log"] = np.log1p(X_train["netmonthlyincome"].clip(lower=0))
    X_test["netmonthlyincome_log"] = np.log1p(X_test["netmonthlyincome"].clip(lower=0))
    X_train = X_train.drop(columns=["netmonthlyincome"])
    X_test = X_test.drop(columns=["netmonthlyincome"])

print("Remaining NaNs in X_train:", X_train.isna().sum().sum())
print("Remaining NaNs in X_test:", X_test.isna().sum().sum())

Remaining NaNs in X_train: 0
Remaining NaNs in X_test: 0


## 6. Preprocessing Pipeline (scale numeric, one-hot encode categorical)

In [ ]:
numeric_columns = X_train.select_dtypes(include=np.number).columns
categorical_columns = X_train.select_dtypes(include="object").columns
print(f"{len(numeric_columns)} numeric columns, {len(categorical_columns)} categorical columns")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print("Processed shapes:", X_train_processed.shape, X_test_processed.shape)

NameError: name 'df' is not defined

In [32]:
def evaluate_regression(model, X_te, y_te):
    y_pred = model.predict(X_te)
    return {
        "r2": r2_score(y_te, y_pred),
        "mae": mean_absolute_error(y_te, y_pred),
        "rmse": root_mean_squared_error(y_te, y_pred),
    }

def evaluate_classification(model, X_te, y_te, binary):
    y_pred = model.predict(X_te)
    metrics = {
        "accuracy": accuracy_score(y_te, y_pred),
        "f1_macro": f1_score(y_te, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_te, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_te, y_pred, average="macro", zero_division=0),
        # Weighted averages
        "precision_weighted": precision_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        # Confusion Matrix
        "confusion_matrix": confusion_matrix(y_te, y_pred)        
    }
    if binary and hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_te)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_te, y_prob)
    return metrics

In [25]:
feature_names

array(['num__prospectid', 'num__total_tl', 'num__tot_closed_tl',
       'num__tot_active_tl', 'num__total_tl_opened_l6m',
       'num__tot_tl_closed_l6m', 'num__pct_tl_open_l6m',
       'num__pct_tl_closed_l6m', 'num__pct_active_tl',
       'num__pct_closed_tl', 'num__total_tl_opened_l12m',
       'num__tot_tl_closed_l12m', 'num__pct_tl_open_l12m',
       'num__pct_tl_closed_l12m', 'num__tot_missed_pmnt', 'num__auto_tl',
       'num__cc_tl', 'num__consumer_tl', 'num__gold_tl', 'num__home_tl',
       'num__pl_tl', 'num__secured_tl', 'num__unsecured_tl',
       'num__other_tl', 'num__age_oldest_tl', 'num__age_newest_tl',
       'num__time_since_recent_payment',
       'num__time_since_first_deliquency',
       'num__time_since_recent_deliquency', 'num__num_times_delinquent',
       'num__max_delinquency_level', 'num__max_recent_level_of_deliq',
       'num__num_deliq_6mts', 'num__num_deliq_12mts',
       'num__num_deliq_6_12mts', 'num__max_deliq_6mts',
       'num__max_deliq_12mts', 'num

## Least-Important Features via Lasso
Lasso's L1 penalty drives the coefficients of uninformative features to exactly zero, which makes it a built-in feature-selection tool: sort by `|coefficient|` and whatever sits near/at zero is what Lasso considers least useful for predicting the target. This is done for the **linear** target (`credit_score`, via `LassoCV`) and for **both logistic** targets (`approved_flag` binary & multiclass, via L1-penalized `LogisticRegressionCV`), since Lasso applies equally to regression and classification.

In [26]:
def get_lasso_ranked_features(coefs, feature_names, zero_tol=1e-4):
    """Return a DataFrame of features sorted ascending by |coefficient| (least
    important first), with a boolean flag for coefficients Lasso zeroed out entirely."""
    ranked = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefs,
        "abs_coefficient": np.abs(coefs),
    })
    ranked["zeroed_out"] = ranked["abs_coefficient"] < zero_tol
    return ranked.sort_values("abs_coefficient", ascending=True).reset_index(drop=True)

In [29]:
# --- Linear regression (credit_score): LassoCV auto-selects alpha via internal CV ---
lasso_cv_linear = LassoCV(cv=3, random_state=RANDOM_STATE, max_iter=500, n_jobs=None)
lasso_cv_linear.fit(X_train_processed, y_linear_train)

print(f"LassoCV selected alpha: {lasso_cv_linear.alpha_:.5f}")

linear_lasso_ranked = get_lasso_ranked_features(lasso_cv_linear.coef_, feature_names)
n_zeroed = linear_lasso_ranked["zeroed_out"].sum()
print(f"{n_zeroed} / {len(feature_names)} features zeroed out by Lasso (credit_score model)")

linear_lasso_ranked.to_csv(f"{RESULTS_DIR}/lasso_least_important_features_linear.csv", index=False)
linear_lasso_ranked.head(15)  # 15 LEAST important features

LassoCV selected alpha: 0.01219
52 / 102 features zeroed out by Lasso (credit_score model)


,feature,coefficient,abs_coefficient,zeroed_out
0,cat__first_prod_enq2_others,-0.0,0.0,True
1,cat__education_OTHERS,0.0,0.0,True
2,cat__education_GRADUATE,-0.0,0.0,True
3,num__max_recent_level_of_deliq,0.0,0.0,True
4,num__num_deliq_12mts,-0.0,0.0,True
5,num__num_deliq_6_12mts,-0.0,0.0,True
6,cat__education_12TH,0.0,0.0,True
7,cat__maritalstatus_Single,0.0,0.0,True
8,cat__maritalstatus_Married,-0.0,0.0,True
9,num__num_times_60p_dpd,-0.0,0.0,True


In [33]:
selected_features = linear_lasso_ranked.loc[
    ~linear_lasso_ranked["zeroed_out"],
    "feature"
].tolist()

print(f"Selected features: {len(selected_features)}")
print(f"Removed features: {len(feature_names) - len(selected_features)}")

X_train_lasso_selected = X_train_processed[selected_features]
X_test_lasso_selected = X_test_processed[selected_features]

model = LinearRegression()
model.fit(X_train_lasso_selected, y_linear_train)
results = evaluate_regression(model, X_test_lasso_selected, y_linear_test)
print(results)

Selected features: 50
Removed features: 52
{'r2': 0.9106074026831211, 'mae': 5.318284493369379, 'rmse': 6.114924658828917}


In [34]:
# --- Logistic regression, binary (approved_flag P1/P2 vs P3/P4): L1 LogisticRegressionCV ---
logreg_cv_binary = LogisticRegressionCV(
    Cs=10, cv=5, l1_ratios=[1.0], solver="saga",
    max_iter=3000, random_state=RANDOM_STATE, scoring="f1_macro",
)
logreg_cv_binary.fit(X_train_processed, y_bin_train)

print(f"LogisticRegressionCV (binary) selected C: {logreg_cv_binary.C_[0]:.5f}")

binary_lasso_ranked = get_lasso_ranked_features(logreg_cv_binary.coef_.ravel(), feature_names)
n_zeroed = binary_lasso_ranked["zeroed_out"].sum()
print(f"{n_zeroed} / {len(feature_names)} features zeroed out by L1 logistic (binary model)")

binary_lasso_ranked.to_csv(f"{RESULTS_DIR}/lasso_least_important_features_logistic_binary.csv", index=False)
binary_lasso_ranked.head(15)

/home/neeraj/Projects/ml_course/ml_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2150: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


TypeError: sklearn.metrics._scorer.make_scorer() got multiple values for keyword argument 'pos_label'

In [ ]:
# --- Logistic regression, multiclass (P1/P2/P3/P4): one coefficient vector per class,
# so a feature's overall importance is summarized as the L2 norm across classes ---
logreg_cv_multi = LogisticRegressionCV(
    Cs=10, cv=5, l1_ratios=[1.0], solver="saga",
    max_iter=3000, random_state=RANDOM_STATE, scoring="f1_macro",
)
logreg_cv_multi.fit(X_train_processed, y_multi_train)

print(f"LogisticRegressionCV (multiclass) selected C per class: {dict(zip(logreg_cv_multi.classes_, logreg_cv_multi.C_))}")

multi_coef_norm = np.linalg.norm(logreg_cv_multi.coef_, axis=0)  # combine across classes
multi_lasso_ranked = get_lasso_ranked_features(multi_coef_norm, feature_names)
n_zeroed = multi_lasso_ranked["zeroed_out"].sum()
print(f"{n_zeroed} / {len(feature_names)} features zeroed out across ALL classes (multiclass model)")

multi_lasso_ranked.to_csv(RESULTS_DIR / "lasso_least_important_features_logistic_multiclass.csv", index=False)
multi_lasso_ranked.head(15)

In [ ]:
# --- Combined view: features Lasso considers unimportant across all three targets ---
common_unimportant = set(linear_lasso_ranked.loc[linear_lasso_ranked["zeroed_out"], "feature"]) \
    & set(binary_lasso_ranked.loc[binary_lasso_ranked["zeroed_out"], "feature"]) \
    & set(multi_lasso_ranked.loc[multi_lasso_ranked["zeroed_out"], "feature"])

print(f"{len(common_unimportant)} features zeroed out by Lasso in ALL THREE models "
      f"(safe candidates to drop):")
sorted(common_unimportant)